# Grid run: models × methods over the persona prompts (CIS 3194)

Runs every combination of **model** (Mistral, Qwen) and **method** on the prompts in `prompts.jsonl`
(persona × question × language × option order), and saves one file per model × method.

Methods (three ways to get the model's answer distribution for a prompt):

* `prob`: no text is generated. One forward pass gives the probability that the reply starts with each option number; renormalised over the valid options.
* `prompt_greedy`: the model writes its answer with greedy decoding (temperature 0). One answer per prompt, so the distribution is a single option.
* `prompt`: the model writes its answer `N_SAMPLES` times with sampling (`TEMPERATURE`, `TOP_K`, `TOP_P`). The distribution is the share of times each option was written.

For all methods the answers are mapped back to CIS codes, and the option orders are averaged afterwards (`order_tv` = how much the answer depends on the order).
`valid_mass` = for `prob`, probability on valid option numbers; for the generation methods, share of answers that could be read as a valid option.

**Setup:** Runtime → Change runtime type → T4 GPU. Edit the config cell, then run all cells. Results are saved in Google Drive: finished model × method files are skipped, and an unfinished one resumes from its last checkpoint, so after a disconnection you can run the cells again (config, prompts, helpers, model helpers, grid) and it continues.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

### Save results in Google Drive
Colab erases its files when the runtime restarts. Mounting Drive makes every finished file survive (accept the permission request).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODELS = {
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # France
    "qwen":    "Qwen/Qwen2.5-7B-Instruct",             # China
}
RUN_MODELS = ["qwen", "mistral"]                       # the grid: which models ...
METHODS = ["prob", "prompt_greedy", "prompt"]          # ... and which methods
ORDERS = "two"        # "two" (original + reversed) or "cyclic" (all rotations; prompts_cyclic.jsonl)

# settings of the sampling method `prompt` (TOP_K = 0 and TOP_P = 1.0 mean no truncation,
# so that with TEMPERATURE = 1.0 the samples follow the model's own probabilities, like `prob`)
N_SAMPLES = 30
SAMPLE_BATCH = 10     # samples generated at once (lower it if the GPU runs out of memory)
TEMPERATURE = 1.0
TOP_K = 0
TOP_P = 1.0
MAX_NEW_TOKENS = 6    # the answer is just a number
SEED = 0

PROMPTS_PATH = "prompts_cyclic.jsonl" if ORDERS == "cyclic" else "prompts.jsonl"
OUT_DIR = "/content/drive/MyDrive/llm_grid_results" + ("_cyclic" if ORDERS == "cyclic" else "")   # in your Google Drive

In [ ]:
import json, os
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(PROMPTS_PATH):
    from google.colab import files
    files.upload()   # choose data/processed/<PROMPTS_PATH>

with open(PROMPTS_PATH, encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f]
assert all("number_to_code" in p for p in prompts), "this notebook needs the numbered prompts (not the letters version)"
print(len(prompts), "prompts from", PROMPTS_PATH)
print("grid:", len(RUN_MODELS), "models x", len(METHODS), "methods =", len(RUN_MODELS) * len(METHODS), "runs of", len(prompts), "prompts")

### Helpers that do not need the model
`parse_answer` reads the option number at the start of a written answer and maps it to the CIS code (anything else counts as invalid).
`to_distribution` turns a list of parsed answers into a distribution over the CIS codes of the question.

In [ ]:
import re
import numpy as np
import pandas as pd


def parse_answer(text, mapping):
    # "3", " 3.", "3. Agree" -> CIS code of option 3; None if the answer does not start with a valid option number
    m = re.match(r"\s*(\d+)", text)
    return mapping.get(m.group(1)) if m else None


def to_distribution(parsed, mapping):
    # list of CIS codes (None = invalid answer) -> ({code: share among valid answers}, valid_mass)
    codes = sorted(set(mapping.values()))
    valid = [c for c in parsed if c is not None]
    valid_mass = len(valid) / len(parsed)
    if not valid:
        return {c: np.nan for c in codes}, valid_mass
    return {c: valid.count(c) / len(valid) for c in codes}, valid_mass


def average_orders(prompt_level, model_id):
    # one row per persona x item x language; p_1...p_6 = proportions by CIS code (average over option orders)
    by_order = prompt_level.pivot_table(index=["persona_id", "item", "lang", "code"], columns="order", values="proportion")
    by_order["proportion"] = by_order.mean(axis=1)
    wide = by_order["proportion"].unstack("code")
    wide.columns = [f"p_{c}" for c in wide.columns]

    # order_tv = 2 x mean over orders of TV(order distribution, average distribution)
    orders = by_order.drop(columns="proportion")
    dev = orders.sub(by_order["proportion"], axis=0).abs()
    tv_per_order = dev.groupby(level=["persona_id", "item", "lang"]).sum() / 2
    wide["order_tv"] = 2 * tv_per_order.mean(axis=1)

    mass = prompt_level.drop_duplicates("prompt_id").pivot_table(index=["persona_id", "item", "lang"], columns="order", values="valid_mass")
    wide["valid_mass_original"] = mass["original"]
    wide["valid_mass_reversed"] = mass["reversed"]
    wide["valid_mass_min"] = mass.min(axis=1)
    wide["n_orders"] = mass.notna().sum(axis=1)
    out = wide.reset_index()
    out.insert(0, "model", model_id)
    return out

### Model-dependent helpers
`load_model` loads one model in 8 bit and finds the vocabulary tokens that stand for each option number (needed by `prob`).
`method_prob`, `method_greedy` and `method_sample` take one prompt record and return `(distribution by CIS code, valid_mass)`.

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = model = None
LABEL_IDS = SPACE_ID = None


def label_to_code(record):
    return record["number_to_code"]


def load_model(model_id):
    global tok, model, LABEL_IDS, SPACE_ID
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="auto", torch_dtype=torch.float16
    )
    model.eval()
    labels = sorted({L for p in prompts for L in label_to_code(p)})
    decoded = tok.batch_decode([[i] for i in range(len(tok))])
    LABEL_IDS = {L: [i for i, s in enumerate(decoded) if s.strip() == L] for L in labels}
    SPACE_ID = next(tok.convert_tokens_to_ids(t) for t in ("▁", "Ġ") if t in tok.get_vocab())
    assert all(LABEL_IDS.values()), "some option label has no token"
    print(f"loaded {model_id}; GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def free_model():
    global tok, model
    tok = model = None
    gc.collect()
    torch.cuda.empty_cache()


def chat_ids(prompt):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    return enc["input_ids"]


@torch.no_grad()
def method_prob(record):
    # P(reply starts with label L), renormalised over the valid labels
    mapping = label_to_code(record)
    ids = torch.cat([chat_ids(record["prompt"]), torch.tensor([[SPACE_ID]])], dim=1).to(model.device)
    p_first, p_after_space = torch.softmax(model(ids).logits[0, -2:].float(), dim=-1)
    raw = {L: (p_first[LABEL_IDS[L]].sum() + p_first[SPACE_ID] * p_after_space[LABEL_IDS[L]].sum()).item() for L in mapping}
    valid_mass = sum(raw.values())
    return {mapping[L]: p / valid_mass for L, p in raw.items()}, valid_mass


@torch.no_grad()
def generate_texts(prompt, n, sample):
    ids = chat_ids(prompt).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.eos_token_id, attention_mask=torch.ones_like(ids),
              repetition_penalty=1.0)   # Qwen's default generation settings use 1.05; we want the plain model
    if sample:
        kw.update(do_sample=True, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P, num_return_sequences=n)
    else:
        kw.update(do_sample=False, temperature=None, top_k=None, top_p=None)
    out = model.generate(ids, **kw)
    return tok.batch_decode(out[:, ids.shape[1]:], skip_special_tokens=True)


def method_greedy(record):
    mapping = label_to_code(record)
    texts = generate_texts(record["prompt"], 1, sample=False)
    return to_distribution([parse_answer(t, mapping) for t in texts], mapping)


def method_sample(record):
    mapping = label_to_code(record)
    texts = []
    for start in range(0, N_SAMPLES, SAMPLE_BATCH):
        texts += generate_texts(record["prompt"], min(SAMPLE_BATCH, N_SAMPLES - start), sample=True)
    return to_distribution([parse_answer(t, mapping) for t in texts], mapping)


METHOD_FUNCS = {"prob": method_prob, "prompt_greedy": method_greedy, "prompt": method_sample}

### Quick check before the grid
Looks at one prompt with each method and prints what the model writes, so you can see that the answers are read correctly (the printed raw texts should start with an option number). Load a model first (this cell loads the first model of `RUN_MODELS`).

In [ ]:
load_model(MODELS[RUN_MODELS[0]])
example = prompts[1]
print(example["prompt_id"], example["number_to_code"])
print("raw greedy text:", generate_texts(example["prompt"], 1, sample=False))
print("raw sampled texts:", generate_texts(example["prompt"], 5, sample=True))
for name, fn in METHOD_FUNCS.items():
    dist, vm = fn(example)
    print(f"{name:14s} valid_mass={vm:.3f}", {k: round(v, 3) for k, v in dist.items()})
free_model()

### The grid
For each model: load it once, run the methods that are not done yet, save one prompt-level CSV per method in `OUT_DIR`, free the GPU. If Colab disconnects, run the config cell and this cell again: finished model × method files are skipped.

In [ ]:
from tqdm.auto import tqdm


def out_path(model_key, method):
    return f"{OUT_DIR}/prompt_level_{MODELS[model_key].split('/')[-1]}__{method}.csv"


for model_key in RUN_MODELS:
    pending = [m for m in METHODS if not os.path.exists(out_path(model_key, m))]
    if not pending:
        print(model_key, "all done, skipping")
        continue
    load_model(MODELS[model_key])
    for method in pending:
        torch.manual_seed(SEED)
        partial = out_path(model_key, method) + ".partial"      # progress file, so a restart does not lose the method
        rows, done = [], set()
        if os.path.exists(partial):
            rows = pd.read_csv(partial).to_dict("records")
            done = {r["prompt_id"] for r in rows}
            print(f"{model_key} / {method}: resuming, {len(done)} of {len(prompts)} prompts already done")
        for i, r in enumerate(tqdm(prompts, desc=f"{model_key} / {method}")):
            if r["prompt_id"] in done:
                continue
            dist, valid_mass = METHOD_FUNCS[method](r)
            for code_, p in dist.items():
                rows.append({"prompt_id": r["prompt_id"], "persona_id": r["persona_id"], "item": r["item"], "lang": r["lang"],
                             "order": r["order"], "code": code_, "proportion": p, "valid_mass": valid_mass})
            if (i + 1) % 25 == 0:
                pd.DataFrame(rows).to_csv(partial, index=False)
        df = pd.DataFrame(rows)
        df.insert(0, "method", method)
        df.insert(0, "model", MODELS[model_key])
        df.to_csv(out_path(model_key, method), index=False)
        if os.path.exists(partial):
            os.remove(partial)
        print(model_key, method, "saved; mean valid_mass =", round(df.drop_duplicates("prompt_id").valid_mass.mean(), 3))
    free_model()

### Put everything in one table
One row per **model × method × persona × item × language**, with the option orders averaged. This is the grid table that is analysed on the computer.

In [ ]:
import glob

parts = []
for path in sorted(glob.glob(f"{OUT_DIR}/prompt_level_*.csv")):
    pl = pd.read_csv(path)
    d = average_orders(pl, pl["model"].iloc[0])
    d.insert(1, "method", pl["method"].iloc[0])
    parts.append(d)
grid = pd.concat(parts, ignore_index=True)
grid.to_csv(f"{OUT_DIR}/grid_results.csv", index=False)
print(len(grid), "rows;", grid.groupby(["model", "method"]).size().to_dict())
print("Mean order effect (TV) by method x item:")
print(grid.groupby(["method", "item"])["order_tv"].mean().round(2).unstack())
print("Mean valid_mass by method:")
print(grid.groupby("method")["valid_mass_min"].mean().round(3))

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("grid_results_download", "zip", OUT_DIR)
files.download("grid_results_download.zip")